In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
pd.set_option("display.max_columns", 50)

In [3]:
hc = pd.read_csv(RAW / "application_train.csv")   # Home Credit
cr = pd.read_csv(RAW / "credit_risk_dataset1.csv")  # dataset de ton camarade
print(hc.shape, cr.shape)

(307511, 122) (32581, 12)


In [4]:
# 1. Cible
print(hc["TARGET"].value_counts(normalize=True).round(4))
print(cr["loan_status"].value_counts(normalize=True).round(4))

# 2. Types de colonnes
print(hc.dtypes.value_counts())
print(cr.dtypes)

# 3. Valeurs manquantes
manq_hc = hc.isna().mean().sort_values(ascending=False)
print((manq_hc.head(15) * 100).round(1))
print("Colonnes HC avec plus de 50 % de manquants :", (manq_hc > 0.5).sum())
print((cr.isna().mean() * 100).round(1))

# 4. Doublons
print("Doublons HC :", hc.duplicated().sum(), "| autre jeu :", cr.duplicated().sum())
print("SK_ID_CURR unique :", hc["SK_ID_CURR"].is_unique)

# 5. Valeurs suspectes
print("DAYS_EMPLOYED = 365243 :", (hc["DAYS_EMPLOYED"] == 365243).sum())
print((-hc["DAYS_BIRTH"] / 365).describe().round(1))
print(hc["CODE_GENDER"].value_counts())
print(cr[["person_age", "person_emp_length", "person_income"]].describe().round(1))
print("âge > 80 :", (cr["person_age"] > 80).sum(), "| ancienneté > 60 :", (cr["person_emp_length"] > 60).sum())

TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64
loan_status
0    0.7818
1    0.2182
Name: proportion, dtype: float64
float64    65
int64      41
str        16
Name: count, dtype: int64
person_age                      int64
person_income                   int64
person_home_ownership             str
person_emp_length             float64
loan_intent                       str
loan_grade                        str
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
dtype: object
COMMONAREA_AVG              69.9
COMMONAREA_MODE             69.9
COMMONAREA_MEDI             69.9
NONLIVINGAPARTMENTS_MEDI    69.4
NONLIVINGAPARTMENTS_MODE    69.4
NONLIVINGAPARTMENTS_AVG     69.4
FONDKAPREMONT_MODE          68.4
LIVINGAPARTMENTS_AVG        68.4
LIVINGAPARTMENTS_MEDI       68.4
LIVINGAPARTMENTS_MODE       68.4

In [5]:
desc = pd.read_csv(RAW / "HomeCredit_columns_description.csv", encoding="latin-1")
print(desc.columns.tolist())
desc.head()

['Unnamed: 0', 'Table', 'Row', 'Description', 'Special']


,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


In [6]:
desc_app = desc[desc["Table"].str.contains("application", na=False)]
desc_app = desc_app.rename(columns={"Row": "colonne"})[["colonne", "Description", "Special"]]

profil = pd.DataFrame({
    "colonne": hc.columns,
    "type": hc.dtypes.astype(str).values,
    "pct_manquant": (hc.isna().mean() * 100).round(1).values,
    "nb_valeurs_distinctes": hc.nunique().values,
    "exemples": [hc[c].dropna().unique()[:5].tolist() for c in hc.columns],
}).merge(desc_app, on="colonne", how="left")

profil.to_csv("../data/interim/profil_home_credit.csv", index=False)
print(profil.shape)

(130, 7)


In [7]:
for c in hc.select_dtypes(exclude="number").columns:
    print(f"\n{c}\n", hc[c].value_counts(dropna=False).head(12))


NAME_CONTRACT_TYPE
 NAME_CONTRACT_TYPE
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64

CODE_GENDER
 CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

FLAG_OWN_CAR
 FLAG_OWN_CAR
N    202924
Y    104587
Name: count, dtype: int64

FLAG_OWN_REALTY
 FLAG_OWN_REALTY
Y    213312
N     94199
Name: count, dtype: int64

NAME_TYPE_SUITE
 NAME_TYPE_SUITE
Unaccompanied      248526
Family              40149
Spouse, partner     11370
Children             3267
Other_B              1770
NaN                  1292
Other_A               866
Group of people       271
Name: count, dtype: int64

NAME_INCOME_TYPE
 NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
Pensioner                55362
State servant            21703
Unemployed                  22
Student                     18
Businessman                 10
Maternity leave              5
Name: count, dtype: int64

NAME_EDUCATION_TYPE
 NAME_EDUCATION_TYPE
Second

In [8]:
import sys
sys.path.append("..")
from src.cleaning import nettoyer_home_credit, nettoyer_credit_risk

hc_clean = nettoyer_home_credit(hc)
cr_clean = nettoyer_credit_risk(cr)
print(hc_clean.shape, cr_clean.shape)

PROCESSED = Path("../data/processed")
hc_clean.to_parquet(PROCESSED / "home_credit_clean.parquet", index=False)
cr_clean.to_parquet(PROCESSED / "credit_risk_clean.parquet", index=False)

HC : 29279 lignes hors 'Cash loans' retirées
HC : 52008 lignes avec DAYS_EMPLOYED = 365243
CR : 165 doublons retirés
CR : 5 lignes avec âge > 100 retirées
CR : 2 lignes avec ancienneté impossible retirées
(278232, 18) (32409, 10)
